# Huấn luyện NLU: PhoBERT — Joint Intent Classification + Slot Filling

**Đồ án Xử lý ngôn ngữ tự nhiên — Đề tài 4: Trợ lý ảo tư vấn tuyển sinh/đào tạo.** Đây là *mã nguồn huấn luyện* (sản phẩm nộp số 2).

Mô hình nhận một câu hỏi tiếng Việt và cùng lúc trả về:
- **Ý định (intent)** — 7 lớp: `hoi_diem_chuan`, `hoi_dieu_kien_tuyen_sinh`, `hoi_hoc_phi`, `hoi_quy_che_hoc_vu`, `tu_van_lo_trinh`, `chao_hoi`, `ngoai_pham_vi`.
- **Thực thể (slot)** gán nhãn BIO theo từ: `nganh_hoc`, `nam`, `phuong_thuc`.

Kiến trúc kiểu **JointBERT**: bộ mã hoá `vinai/phobert-base` (RoBERTa huấn luyện trên 20 GB văn bản tiếng Việt đã tách từ) + 2 đầu phân loại tuyến tính — đầu intent dùng vector của token `<s>`, đầu slot dùng vector của **subword đầu tiên** mỗi từ. Hàm mất mát: $\mathcal{L} = \mathcal{L}_{intent} + \mathcal{L}_{slot}$ (cross-entropy, bỏ qua subword không phải đầu từ).

Toàn bộ tiền xử lý và định nghĩa mô hình **import từ chính mã nguồn web app** (`app/services/text_service.py`, `app/services/nlu_model.py`) — train và suy luận dùng chung một đường code.

**Chạy trên Colab:** tải thư mục `app/`, `data/nlu/` và file `.env` (hoặc `.env.example` đổi tên) lên cùng thư mục notebook, `pip install transformers pyvi seqeval pydantic-settings sqlalchemy`, chọn GPU runtime.

## 1. Thiết lập

In [1]:
import json
import random
import sys
import time
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from seqeval.metrics import classification_report, f1_score
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from app.config import Settings
from app.services import nlu_model
from app.services.nlu_service import NluService
from app.services.text_service import segment_lowercased, spans_to_bio

settings = Settings(_env_file=ROOT / ".env")
SEED = 42
EPOCHS = 20
PATIENCE = 4
BATCH_SIZE = 16
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_SHARE = 0.1
DATA_DIR = ROOT / "data" / "nlu"
OUTPUT_DIR = ROOT / settings.nlu_model_dir

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
print("Mô hình nền:", settings.nlu_base_model, "| torch", torch.__version__)

Thiết bị: cuda NVIDIA GeForce RTX 3050 6GB Laptop GPU
Mô hình nền: vinai/phobert-base | torch 2.11.0+cu128


## 2. Dữ liệu và khảo sát (EDA)

Bộ dữ liệu tự xây (`training/build_dataset.py`): mẫu câu viết tay × giá trị thật (109 tên ngành lấy từ bảng học phí Đại học Cần Thơ 2026 đã cào bằng Brightdata, năm 2019–2026, 10 cách gọi phương thức xét tuyển); ~15% câu bỏ dấu, 50% câu chữ thường; câu ngoài phạm vi có câu "bẫy" chứa từ giống tuyển sinh ("điểm số trận…", "học phí khoá lái xe…"). Chia 80/10/10 theo từng intent, không trùng câu giữa các tập.

In [2]:
labels = json.loads((DATA_DIR / "labels.json").read_text(encoding="utf-8"))
splits = {
    name: [json.loads(line) for line in (DATA_DIR / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()]
    for name in ("train", "dev", "test")
}
for name, rows in splits.items():
    print(f"{name:5} {len(rows):5} câu", dict(sorted(Counter(row["intent"] for row in rows).items())))
lengths = [len(row["text"].split()) for rows in splits.values() for row in rows]
print("Độ dài câu (âm tiết): min", min(lengths), "| trung bình", round(sum(lengths) / len(lengths), 1), "| max", max(lengths))
print("Số thực thể:", dict(Counter(entity["label"] for rows in splits.values() for row in rows for entity in row["entities"])))
for row in splits["train"][:5]:
    print(row)

train  1332 câu {'chao_hoi': 89, 'hoi_diem_chuan': 144, 'hoi_dieu_kien_tuyen_sinh': 140, 'hoi_hoc_phi': 119, 'hoi_quy_che_hoc_vu': 184, 'ngoai_pham_vi': 537, 'tu_van_lo_trinh': 119}
dev     164 câu {'chao_hoi': 11, 'hoi_diem_chuan': 18, 'hoi_dieu_kien_tuyen_sinh': 17, 'hoi_hoc_phi': 14, 'hoi_quy_che_hoc_vu': 23, 'ngoai_pham_vi': 67, 'tu_van_lo_trinh': 14}
test    173 câu {'chao_hoi': 12, 'hoi_diem_chuan': 18, 'hoi_dieu_kien_tuyen_sinh': 19, 'hoi_hoc_phi': 16, 'hoi_quy_che_hoc_vu': 24, 'ngoai_pham_vi': 68, 'tu_van_lo_trinh': 16}
Độ dài câu (âm tiết): min 1 | trung bình 10.2 | max 22
Số thực thể: {'nganh_hoc': 426, 'nam': 150, 'phuong_thuc': 78}
{'text': 'cho mình hỏi cho em hỏi điểm chuẩn Hệ thống thông tin năm 2023 ạ', 'intent': 'hoi_diem_chuan', 'entities': [{'start': 35, 'end': 53, 'label': 'nganh_hoc'}, {'start': 58, 'end': 62, 'label': 'nam'}]}
{'text': 'ngành Văn học bao nhiêu điểm năm 2022 vậy?', 'intent': 'hoi_diem_chuan', 'entities': [{'start': 6, 'end': 13, 'label': 'nganh_hoc

## 3. Tiền xử lý tiếng Việt

1. **Chuẩn hoá Unicode NFC** + gộp khoảng trắng (câu gõ trên điện thoại/Windows có thể ở dạng NFD — nhìn giống nhau nhưng khác mã).
2. **Chữ thường khi đưa vào model** (`segment_lowercased`): vị trí ký tự giữ nguyên nên slot vẫn trích đúng chữ hoa/thường gốc; model không thể dùng chữ hoa đầu câu làm "lối tắt" (xem mục 8).
3. **Tách từ** bằng `pyvi` (PhoBERT được huấn luyện trên văn bản đã tách từ, các âm tiết của một từ nối bằng `_`: `Hệ_thống`).
4. Nhãn thực thể lưu theo **vị trí ký tự** trong câu gốc, sau khi tách từ mới đổi sang **BIO theo từ** — không phụ thuộc cách tách từ.
5. Mỗi từ tách thành subword BPE; chỉ **subword đầu** mang nhãn slot, các subword còn lại bỏ qua khi tính loss (`-100`).

In [3]:
example = splits["train"][0]
tokens = segment_lowercased(example["text"])
print(example["text"])
for token, label in zip(tokens, spans_to_bio(tokens, example["entities"])):
    print(f"  {token.word:28} {label}")

cho mình hỏi cho em hỏi điểm chuẩn Hệ thống thông tin năm 2023 ạ
  cho                          O
  mình                         O
  hỏi                          O
  cho                          O
  em                           O
  hỏi                          O
  điểm_chuẩn                   O
  hệ_thống                     B-nganh_hoc
  thông_tin                    I-nganh_hoc
  năm                          O
  2023                         B-nam
  ạ                            O


In [4]:
tokenizer = AutoTokenizer.from_pretrained(settings.nlu_base_model)
intent_id = {name: index for index, name in enumerate(labels["intents"])}
slot_id = {name: index for index, name in enumerate(labels["slot_labels"])}


def featurize(row):
    tokens = segment_lowercased(row["text"])
    encoded = nlu_model.encode_words(tokenizer, [token.word for token in tokens], nlu_model.MAX_LEN)
    word_labels = spans_to_bio(tokens, row["entities"])[:len(encoded.first_positions)]
    slot_targets = [nlu_model.IGNORE_INDEX] * len(encoded.input_ids)
    for position, label in zip(encoded.first_positions, word_labels):
        slot_targets[position] = slot_id[label]
    return {"text": row["text"], "encoded": encoded, "intent": intent_id[row["intent"]], "slots": slot_targets, "word_labels": word_labels}


def collate(batch):
    input_ids, attention_mask = nlu_model.pad_batch([item["encoded"] for item in batch], tokenizer.pad_token_id)
    slots = torch.full(input_ids.shape, nlu_model.IGNORE_INDEX, dtype=torch.long)
    for row, item in enumerate(batch):
        slots[row, :len(item["slots"])] = torch.tensor(item["slots"])
    return input_ids, attention_mask, torch.tensor([item["intent"] for item in batch]), slots


features = {name: [featurize(row) for row in rows] for name, rows in splits.items()}
train_loader = DataLoader(features["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                          generator=torch.Generator().manual_seed(SEED))
print({name: len(items) for name, items in features.items()}, "| độ dài subword tối đa:",
      max(len(item["encoded"].input_ids) for items in features.values() for item in items))

{'train': 1332, 'dev': 164, 'test': 173} | độ dài subword tối đa: 28


## 4. Mô hình và huấn luyện

AdamW (lr 5e-5, weight decay 0.01), warmup tuyến tính 10%, fp16 (autocast) trên GPU, cắt gradient 1.0. **Dừng sớm** khi độ chính xác cấp câu trên tập dev không tăng sau 4 epoch; giữ trọng số tốt nhất.

In [5]:
encoder = AutoModel.from_pretrained(settings.nlu_base_model)
model = nlu_model.JointPhoBERT(encoder, len(labels["intents"]), len(labels["slot_labels"])).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_SHARE * total_steps), total_steps)
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=nlu_model.IGNORE_INDEX)
scaler = torch.amp.GradScaler(enabled=device == "cuda")
print(f"{sum(parameter.numel() for parameter in model.parameters()) / 1e6:.1f} triệu tham số")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


135.0 triệu tham số


In [6]:
def predict_split(name):
    model.eval()
    result = {"text": [], "intent_true": [], "intent_pred": [], "confidence": [], "slot_true": [], "slot_pred": []}
    items = features[name]
    with torch.inference_mode():
        for start in range(0, len(items), BATCH_SIZE):
            batch = items[start:start + BATCH_SIZE]
            input_ids, attention_mask, intents, _ = collate(batch)
            with torch.autocast(device_type=device, enabled=device == "cuda"):
                intent_logits, slot_logits = model(input_ids.to(device), attention_mask.to(device))
            probabilities = intent_logits.float().softmax(-1)
            result["intent_true"] += intents.tolist()
            result["intent_pred"] += probabilities.argmax(-1).tolist()
            result["confidence"] += probabilities.max(-1).values.tolist()
            for item, row in zip(batch, slot_logits.argmax(-1).tolist()):
                result["text"].append(item["text"])
                result["slot_true"].append(item["word_labels"])
                result["slot_pred"].append([labels["slot_labels"][row[position]] for position in item["encoded"].first_positions])
    return result


def scores(result):
    pairs = list(zip(result["intent_true"], result["intent_pred"], result["slot_true"], result["slot_pred"]))
    return {
        "intent_accuracy": sum(true == pred for true, pred, _, _ in pairs) / len(pairs),
        "slot_f1": f1_score(result["slot_true"], result["slot_pred"]),
        "sentence_accuracy": sum(true == pred and st == sp for true, pred, st, sp in pairs) / len(pairs),
    }

In [7]:
history, best, stale, best_state = [], -1.0, 0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    started, total_loss = time.time(), 0.0
    for input_ids, attention_mask, intents, slots in train_loader:
        input_ids, attention_mask, intents, slots = (tensor.to(device) for tensor in (input_ids, attention_mask, intents, slots))
        optimizer.zero_grad()
        with torch.autocast(device_type=device, enabled=device == "cuda"):
            intent_logits, slot_logits = model(input_ids, attention_mask)
            loss = loss_fn(intent_logits, intents) + loss_fn(slot_logits.reshape(-1, slot_logits.shape[-1]), slots.reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    dev = scores(predict_split("dev"))
    history.append({"epoch": epoch, "train_loss": total_loss / len(train_loader), **dev, "seconds": time.time() - started})
    print(f"epoch {epoch:2} | loss {history[-1]['train_loss']:.4f} | dev intent {dev['intent_accuracy']:.4f} | "
          f"slot F1 {dev['slot_f1']:.4f} | câu {dev['sentence_accuracy']:.4f} | {history[-1]['seconds']:.1f}s")
    if dev["sentence_accuracy"] > best:
        best, stale = dev["sentence_accuracy"], 0
        best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f"Dừng sớm sau epoch {epoch} (dev không cải thiện {PATIENCE} epoch).")
            break
model.load_state_dict(best_state)
print("Epoch tốt nhất: độ chính xác câu dev =", round(best, 4))

C:\Users\akira\AppData\Local\Temp\ipykernel_5612\520818039.py:16: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch  1 | loss 2.7490 | dev intent 0.4146 | slot F1 0.0000 | câu 0.4085 | 16.6s


epoch  2 | loss 1.5217 | dev intent 0.7561 | slot F1 0.0274 | câu 0.6098 | 17.1s


epoch  3 | loss 0.6684 | dev intent 0.9268 | slot F1 0.5000 | câu 0.6829 | 17.2s


epoch  4 | loss 0.2909 | dev intent 0.9390 | slot F1 0.6466 | câu 0.8049 | 17.0s


...

epoch  9 | loss 0.0141 | dev intent 0.9878 | slot F1 0.9160 | câu 0.9573 | 17.6s


epoch 10 | loss 0.0099 | dev intent 0.9878 | slot F1 0.8992 | câu 0.9512 | 17.1s


epoch 11 | loss 0.0065 | dev intent 0.9756 | slot F1 0.9302 | câu 0.9512 | 18.0s


epoch 12 | loss 0.0062 | dev intent 0.9817 | slot F1 0.9375 | câu 0.9573 | 18.1s


epoch 13 | loss 0.0051 | dev intent 0.9878 | slot F1 0.9147 | câu 0.9573 | 17.9s
Dừng sớm sau epoch 13 (dev không cải thiện 4 epoch).
Epoch tốt nhất: độ chính xác câu dev = 0.9573


## 5. Đánh giá trên tập test

- **Intent accuracy**: tỷ lệ câu đoán đúng ý định.
- **Slot F1** (seqeval, micro, tính theo thực thể trọn vẹn — đúng cả ranh giới lẫn loại).
- **Sentence accuracy**: đúng cả intent lẫn toàn bộ nhãn slot của câu.
- **Tỷ lệ câu trong phạm vi bị đẩy ra ngoài**: quan trọng vì câu `ngoai_pham_vi` được trả lời bằng một câu cố định — đẩy nhầm câu hỏi hợp lệ ra ngoài là lỗi nặng nhất với người dùng.

⚠️ Tập test được sinh từ cùng bộ mẫu câu với tập train (khác giá trị ngành/năm/cách viết), nên số liệu ở đây đo khả năng khái quát trên *giá trị* và *cách viết*; đánh giá đầu–cuối với câu hỏi viết tay độc lập nằm ở `tests/eval` (T42).

In [8]:
test = predict_split("test")
metrics = scores(test)
print(json.dumps(metrics, indent=2))
print(classification_report(test["slot_true"], test["slot_pred"], digits=4))

{
  "intent_accuracy": 0.9884393063583815,
  "slot_f1": 0.9538461538461539,
  "sentence_accuracy": 0.9710982658959537
}
              precision    recall  f1-score   support

         nam     1.0000    1.0000    1.0000        13
   nganh_hoc     0.9348    0.9348    0.9348        46
 phuong_thuc     1.0000    1.0000    1.0000         6

   micro avg     0.9538    0.9538    0.9538        65
   macro avg     0.9783    0.9783    0.9783        65
weighted avg     0.9538    0.9538    0.9538        65



In [9]:
names = labels["intents"]
matrix = [[0] * len(names) for _ in names]
for true, pred in zip(test["intent_true"], test["intent_pred"]):
    matrix[true][pred] += 1
print("Ma trận nhầm lẫn (hàng = thực tế, cột = dự đoán):")
print(" " * 29 + "".join(f"{index:>5}" for index in range(len(names))))
for index, row in enumerate(matrix):
    print(f"{index} {names[index]:27}" + "".join(f"{count:>5}" for count in row))

per_intent = {}
for index, name in enumerate(names):
    true_positive = matrix[index][index]
    predicted = sum(row[index] for row in matrix)
    actual = sum(matrix[index])
    precision = true_positive / predicted if predicted else 0.0
    recall = true_positive / actual if actual else 0.0
    per_intent[name] = {"precision": precision, "recall": recall, "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0, "support": actual}
    print(f"{name:27} P={precision:.4f} R={recall:.4f} F1={per_intent[name]['f1']:.4f} n={actual}")

Ma trận nhầm lẫn (hàng = thực tế, cột = dự đoán):
                                 0    1    2    3    4    5    6
0 hoi_diem_chuan                18    0    0    0    0    0    0
1 hoi_dieu_kien_tuyen_sinh       0   19    0    0    0    0    0
2 hoi_hoc_phi                    0    0   16    0    0    0    0
3 hoi_quy_che_hoc_vu             0    0    0   23    0    0    1
4 tu_van_lo_trinh                0    0    0    0   16    0    0
5 chao_hoi                       0    0    0    0    0   12    0
6 ngoai_pham_vi                  0    0    0    1    0    0   67
hoi_diem_chuan              P=1.0000 R=1.0000 F1=1.0000 n=18
hoi_dieu_kien_tuyen_sinh    P=1.0000 R=1.0000 F1=1.0000 n=19
hoi_hoc_phi                 P=1.0000 R=1.0000 F1=1.0000 n=16
hoi_quy_che_hoc_vu          P=0.9583 R=0.9583 F1=0.9583 n=24
tu_van_lo_trinh             P=1.0000 R=1.0000 F1=1.0000 n=16
chao_hoi                    P=1.0000 R=1.0000 F1=1.0000 n=12
ngoai_pham_vi               P=0.9853 R=0.9853 F1=0.9853 n=68


In [10]:
out_of_scope = intent_id["ngoai_pham_vi"]


def scope_rates(threshold):
    predicted = [out_of_scope if confidence < threshold else pred for pred, confidence in zip(test["intent_pred"], test["confidence"])]
    inside = [pred for true, pred in zip(test["intent_true"], predicted) if true != out_of_scope]
    outside = [pred for true, pred in zip(test["intent_true"], predicted) if true == out_of_scope]
    return {
        "threshold": threshold,
        "in_scope_pushed_out_rate": sum(pred == out_of_scope for pred in inside) / len(inside),
        "out_of_scope_recall": sum(pred == out_of_scope for pred in outside) / len(outside),
    }


scope = [scope_rates(threshold) for threshold in (0.0, 0.3, settings.nlu_min_confidence, 0.7, 0.9)]
for row in scope:
    print(f"ngưỡng {row['threshold']:.2f} | câu trong phạm vi bị đẩy ra ngoài {row['in_scope_pushed_out_rate']:.4f} | "
          f"bắt đúng câu ngoài phạm vi {row['out_of_scope_recall']:.4f}")

ngưỡng 0.00 | câu trong phạm vi bị đẩy ra ngoài 0.0095 | bắt đúng câu ngoài phạm vi 0.9853
ngưỡng 0.30 | câu trong phạm vi bị đẩy ra ngoài 0.0095 | bắt đúng câu ngoài phạm vi 0.9853
ngưỡng 0.50 | câu trong phạm vi bị đẩy ra ngoài 0.0095 | bắt đúng câu ngoài phạm vi 0.9853
ngưỡng 0.70 | câu trong phạm vi bị đẩy ra ngoài 0.0095 | bắt đúng câu ngoài phạm vi 0.9853
ngưỡng 0.90 | câu trong phạm vi bị đẩy ra ngoài 0.0095 | bắt đúng câu ngoài phạm vi 1.0000


In [11]:
errors = [
    (text, names[true], names[pred], round(confidence, 3), [f"{t}->{p}" for t, p in zip(st, sp) if t != p])
    for text, true, pred, confidence, st, sp in zip(test["text"], test["intent_true"], test["intent_pred"], test["confidence"], test["slot_true"], test["slot_pred"])
    if true != pred or st != sp
]
print(len(errors), "câu sai trên", len(test["text"]))
for error in errors:
    print(error)

5 câu sai trên 173
('cho minh hoi xet diem hoc ba thi nganh kien truc lay may diem a?', 'hoi_diem_chuan', 'hoi_diem_chuan', 0.965, ['O->I-nganh_hoc'])
('ban oi sinh vien duoc no bao nhieu tin chi a', 'hoi_quy_che_hoc_vu', 'ngoai_pham_vi', 0.99, [])
('ban oi nganh attt hoc may nam?', 'tu_van_lo_trinh', 'tu_van_lo_trinh', 0.996, ['O->I-nganh_hoc', 'O->I-nganh_hoc'])
('ra truong nganh Marketing luong bao nhieu vay', 'tu_van_lo_trinh', 'tu_van_lo_trinh', 0.996, ['O->I-nganh_hoc'])
('cho minh hoi hoc boi nhu the nao cho nhanh vay', 'ngoai_pham_vi', 'hoi_quy_che_hoc_vu', 0.831, [])


## 6. Lưu model cho web app

Lưu vào `NLU_MODEL_DIR` (mặc định `artifacts/nlu/`): cấu hình encoder, tokenizer (để app **không phải tải mạng** lúc chạy), trọng số (`model.safetensors`), nhãn và số liệu đánh giá (`metrics.json`).

In [12]:
nlu_model.save_artifacts(model.cpu(), tokenizer, labels, OUTPUT_DIR)
model.to(device)
report = {
    "base_model": settings.nlu_base_model,
    "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    "hyperparameters": {"epochs_max": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
                        "weight_decay": WEIGHT_DECAY, "warmup_share": WARMUP_SHARE, "max_len": nlu_model.MAX_LEN, "seed": SEED},
    "dataset": {name: len(rows) for name, rows in splits.items()},
    "history": history,
    "test": metrics,
    "per_intent": per_intent,
    "confusion_matrix": {"labels": names, "matrix": matrix},
    "scope": scope,
    "test_errors": len(errors),
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu:", sorted(path.name for path in OUTPUT_DIR.iterdir()))

Đã lưu: ['added_tokens.json', 'bpe.codes', 'config.json', 'labels.json', 'metrics.json', 'model.safetensors', 'tokenizer_config.json', 'vocab.txt']


## 7. Thử nhanh bằng `NluService` — đúng đường code web app dùng

Các câu dưới đây **viết tay, không có trong bộ dữ liệu**; câu có độ tin cậy dưới `NLU_MIN_CONFIDENCE` được coi là ngoài phạm vi.

In [13]:
service = NluService.load(OUTPUT_DIR, settings.nlu_min_confidence, device)
for question in [
    "Điểm chuẩn ngành Kỹ thuật phần mềm năm 2025 là bao nhiêu?",
    "hoc phi nganh cong nghe thong tin mot nam bao nhieu",
    "Khi nào sinh viên bị buộc thôi học?",
    "Em thích làm game thì nên học ngành gì?",
    "Xét học bạ vào ngành Marketing cần điều kiện gì?",
    "Xin chào trợ lý nhé",
    "Cho mình hỏi giá bitcoin hôm nay",
    "Điểm số trận Việt Nam gặp Thái Lan tối qua",
]:
    result = service.predict(question)
    print(f"{result.intent:26} {result.confidence:.3f} {result.slots}  ← {question}")

hoi_diem_chuan             0.997 {'nganh_hoc': 'Kỹ thuật phần mềm', 'nam': '2025'}  ← Điểm chuẩn ngành Kỹ thuật phần mềm năm 2025 là bao nhiêu?
hoi_hoc_phi                0.996 {'nganh_hoc': 'cong nghe thong tin'}  ← hoc phi nganh cong nghe thong tin mot nam bao nhieu
hoi_quy_che_hoc_vu         0.998 {}  ← Khi nào sinh viên bị buộc thôi học?
tu_van_lo_trinh            0.997 {}  ← Em thích làm game thì nên học ngành gì?


hoi_dieu_kien_tuyen_sinh   0.996 {'phuong_thuc': 'học bạ vào', 'nganh_hoc': 'Marketing'}  ← Xét học bạ vào ngành Marketing cần điều kiện gì?
chao_hoi                   0.997 {}  ← Xin chào trợ lý nhé
ngoai_pham_vi              0.998 {}  ← Cho mình hỏi giá bitcoin hôm nay
ngoai_pham_vi              0.998 {}  ← Điểm số trận Việt Nam gặp Thái Lan tối qua


## 8. Lịch sử cải tiến dữ liệu (thực nghiệm)

Mỗi vòng: huấn luyện → kiểm bằng **câu viết tay không có trong dữ liệu** (`tests/integration/test_nlu_real_model.py`, có test chặn rò rỉ) → tìm nguyên nhân gốc → sửa **dữ liệu**, không chỉnh ngưỡng cho khớp.

| Vòng | Dữ liệu | Test: intent / slot F1 / câu | Lỗi phát hiện trên câu viết tay | Nguyên nhân gốc → cách sửa |
|---|---|---|---|---|
| v1 | 1.317 câu, 15% không dấu | 98,53% / 94,12% / 96,32% | "khi nao bi canh bao hoc vu" → ngoài phạm vi; "Hôm nay giá xăng bao nhiêu?" → học phí | (1) "bao nhiêu" chỉ xuất hiện ở câu học phí/điểm chuẩn; (2) câu không dấu hiếm → thêm câu ngoài phạm vi dạng "… bao nhiêu / khi nào / là gì / như thế nào", tăng câu không dấu lên 35% |
| v2 | 1.669 câu, 35% không dấu | 99,42% / 92,52% / 95,95% | "Hôm nay giá xăng bao nhiêu?" vẫn → học phí, nhưng viết thường thì đúng | (3) câu ngoài phạm vi luôn viết thường, không có "?" → đưa chữ thường vào model, thêm đuôi "?"/"ạ"/"vậy" cho câu ngoài phạm vi |
| v3 | như v2 + đuôi câu | (xem mục 5) | (xem `tests/integration/test_nlu_real_model.py`) | — |

Số liệu từng vòng: `docs/report/nlu_metrics_v1.json`, `nlu_metrics_v2.json`, `artifacts/nlu/metrics.json` (v3).